In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [2]:
os.makedirs("../models", exist_ok=True)
print("Models folder created successfully.")

Models folder created successfully.


In [3]:
DATA_PATH = "../data/processed/final_processed_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (52930, 24)


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating,ContinentId,RegionId,CountryId,...,AttractionAddress,AttractionType,CityId_city,CityName,CountryId_city,Country,RegionId_country,Region,ContinentId_region,Continent
0,3,70456,2022,10,2,640,5,5,21,163,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,United Kingdom,21,Western Europe,5,Europe
1,8,7567,2022,10,4,640,5,2,8,48,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Canada,8,Northern America,2,America
2,9,79069,2022,10,3,640,5,2,9,54,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Brazil,9,South America,2,America
3,10,31019,2022,10,3,640,3,5,17,135,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Switzerland,17,Central Europe,5,Europe
4,15,43611,2022,10,2,640,3,5,21,163,...,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,United Kingdom,21,Western Europe,5,Europe


In [4]:
print("Available Columns:")
print(df.columns.tolist())

Available Columns:
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'CityId_city', 'CityName', 'CountryId_city', 'Country', 'RegionId_country', 'Region', 'ContinentId_region', 'Continent']


In [5]:
TARGET = "Rating"

if TARGET not in df.columns:
    raise ValueError(f"{TARGET} column not found in dataset.")

print("Target variable:", TARGET)
print("Missing values in target:", df[TARGET].isnull().sum())

Target variable: Rating
Missing values in target: 0


In [6]:
df = df.dropna(subset=[TARGET]).copy()

print("Dataset shape after removing missing target values:", df.shape)

Dataset shape after removing missing target values: (52930, 24)


In [7]:
# -----------------------------------
# Final Raw Input Features
# -----------------------------------

feature_columns = [
    "Continent",
    "Region",
    "Country",
    "CityName",
    "AttractionType",
    "VisitMode",
    "VisitYear",
    "VisitMonth"
]

print("Number of selected features:", len(feature_columns))
print("Selected features:")
print(feature_columns)

Number of selected features: 8
Selected features:
['Continent', 'Region', 'Country', 'CityName', 'AttractionType', 'VisitMode', 'VisitYear', 'VisitMonth']


In [8]:
X = df[feature_columns]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (52930, 8)
y shape: (52930,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,   # 20% test data, 80% training data
    random_state=42,
    shuffle=True
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape :", X_test.shape)
print("Training target shape :", y_train.shape)
print("Testing target shape  :", y_test.shape)

print("\nTrain percentage:", round(len(X_train) / len(X) * 100, 2), "%")
print("Test percentage :", round(len(X_test) / len(X) * 100, 2), "%")

Training feature shape: (42344, 8)
Testing feature shape : (10586, 8)
Training target shape : (42344,)
Testing target shape  : (10586,)

Train percentage: 80.0 %
Test percentage : 20.0 %


In [10]:
# -----------------------------------
# Manual Feature Separation
# -----------------------------------

categorical_columns = [
    "Continent",
    "Region",
    "Country",
    "CityName",
    "AttractionType",
    "VisitMode"
]

numerical_columns = [
    "VisitYear",
    "VisitMonth"
]

print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)

Numerical Columns:
['VisitYear', 'VisitMonth']

Categorical Columns:
['Continent', 'Region', 'Country', 'CityName', 'AttractionType', 'VisitMode']


In [11]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("Numerical pipeline created.")

Numerical pipeline created.


In [12]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

print("Categorical pipeline created.")

Categorical pipeline created.


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [14]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )
}

print("Models defined successfully.")
print(list(models.keys()))

Models defined successfully.
['Linear Regression', 'Random Forest', 'XGBoost']


In [15]:
results = []
trained_models = {}

for model_name, model in models.items():
    print("\n" + "=" * 60)
    print(f"Training {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

    trained_models[model_name] = pipeline

    print(f"MAE       : {mae:.4f}")
    print(f"MSE       : {mse:.4f}")
    print(f"RMSE      : {rmse:.4f}")
    print(f"R2 Score  : {r2:.4f}")


Training Linear Regression


MAE       : 0.7262
MSE       : 0.8669
RMSE      : 0.9311
R2 Score  : 0.0795

Training Random Forest
MAE       : 0.7494
MSE       : 0.9625
RMSE      : 0.9811
R2 Score  : -0.0219

Training XGBoost
MAE       : 0.7180
MSE       : 0.8483
RMSE      : 0.9210
R2 Score  : 0.0993


In [16]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

results_df

,Model,MAE,MSE,RMSE,R2 Score
0,XGBoost,0.718008,0.848302,0.921033,0.099291
1,Linear Regression,0.726226,0.866907,0.931078,0.079537
2,Random Forest,0.749389,0.962462,0.981052,-0.021922


In [20]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: XGBoost


In [21]:
joblib.dump(best_pipeline, "../models/rating_model.pkl")
joblib.dump(preprocessor, "../models/rating_preprocessor.pkl")

print("Saved successfully:")
print("- ../models/rating_model.pkl")
print("- ../models/rating_preprocessor.pkl")

Saved successfully:
- ../models/rating_model.pkl
- ../models/rating_preprocessor.pkl


In [22]:
sample_predictions = best_model.predict(X_test.iloc[:10])

prediction_df = pd.DataFrame({
    "Actual Rating": y_test.iloc[:10].values,
    "Predicted Rating": np.round(sample_predictions, 2)
})

prediction_df

,Actual Rating,Predicted Rating
0,5,4.19
1,5,4.19
2,3,3.59
3,4,4.32
4,5,4.15
5,4,4.18
6,5,4.08
7,4,4.10
8,5,4.20
9,5,3.94


In [ ]:
comparison_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_model.predict(X_test)
})

comparison_df.head(20)

,Actual,Predicted
0,5,4.191193
1,5,4.190276
2,3,3.594030
3,4,4.318428
4,5,4.153348
5,4,4.181724
6,5,4.076687
7,4,4.101842
8,5,4.199919
9,5,3.935101


In [23]:
print("=" * 60)
print("Regression Modeling Completed Successfully")
print("=" * 60)

print(f"Best Model      : {best_model_name}")
print(f"Best R2 Score   : {results_df.iloc[0]['R2 Score']:.4f}")
print(f"Best RMSE       : {results_df.iloc[0]['RMSE']:.4f}")

print("\nSaved Files:")
print("1. ../models/rating_model.pkl")
print("2. ../models/rating_preprocessor.pkl")

print("\nThe model is ready for use in the Streamlit application.")

Regression Modeling Completed Successfully
Best Model      : XGBoost
Best R2 Score   : 0.0993
Best RMSE       : 0.9210

Saved Files:
1. ../models/rating_model.pkl
2. ../models/rating_preprocessor.pkl

The model is ready for use in the Streamlit application.


In [24]:
import json

# Save regression metrics for Streamlit
regression_metrics = {
    "Best Model": best_model_name,
    "R2 Score": float(results_df.iloc[0]["R2 Score"]),
    "MAE": float(results_df.iloc[0]["MAE"]),
    "RMSE": float(results_df.iloc[0]["RMSE"])
}

with open("../models/regression_metrics.json", "w") as f:
    json.dump(regression_metrics, f, indent=4)

print("regression_metrics.json saved successfully.")
print(regression_metrics)

regression_metrics.json saved successfully.
{'Best Model': 'XGBoost', 'R2 Score': 0.09929114580154419, 'MAE': 0.7180080413818359, 'RMSE': 0.9210332163040761}


In [25]:
import joblib
import pandas as pd

# Load the saved regression model pipeline
best_pipeline = joblib.load("../models/rating_model.pkl")

# Extract feature importance if supported
if hasattr(best_pipeline.named_steps["model"], "feature_importances_"):
    # Get transformed feature names
    feature_names = (
        best_pipeline.named_steps["preprocessor"]
        .get_feature_names_out()
    )

    # Create feature importance DataFrame
    feature_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": best_pipeline.named_steps["model"].feature_importances_
    }).sort_values("Importance", ascending=False)

    # Save to CSV
    feature_importance_df.to_csv(
        "../models/feature_importance.csv",
        index=False
    )

    print("feature_importance.csv saved successfully.")
    print(feature_importance_df.head(10))

else:
    print("Selected model does not support feature importances.")

feature_importance.csv saved successfully.
                                       Feature  Importance
198            cat__AttractionType_Water Parks    0.161176
188         cat__AttractionType_Historic Sites    0.074163
185                cat__AttractionType_Beaches    0.038698
181                    cat__CityName_N'Djamena    0.023549
23                      cat__Region_South Asia    0.020793
89                      cat__Country_Indonesia    0.020411
187  cat__AttractionType_Flea & Street Markets    0.019482
19                cat__Region_Northern America    0.019232
199             cat__AttractionType_Waterfalls    0.016060
6                        cat__Continent_Europe    0.014638
